# 04 - OOD detection benchmark

Post-hoc detection scores (MSP, Energy) on V0: threshold calibrated on the known intents' validation split, then measured on the known test split and on the unknown intents, split near / far by MASSIVE scenario. Trains nothing; runs on CPU in about a minute.

In [1]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0" anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is inco

In [2]:
%cd /content/drive/MyDrive/incremental_learning_on_the_edge/src
!git pull

/content/drive/MyDrive/incremental_learning_on_the_edge/src
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.75 KiB | 1024 bytes/s, done.
From https://github.com/BrunoSantome/incremental_learning_on_the_edge
   7d3dd50..3025bfa  main       -> origin/main
Updating 7d3dd50..3025bfa
Fast-forward
 src/experiments/ood_benchmark.py | 86 ++++++++++++++++++++++++++++++++++++++++
 1 file changed, 86 insertions(+)


In [ ]:
!python -m experiments.ood_benchmark

2026-10-01 11:28:03.188974: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 20.8kB [00:00, 69.2MB/s]
tokenizer.json: 2.13MB [00:00, 154MB/s]
special_tokens_map.json: 100% 694/694 [00:00<00:00, 6.01MB/s]
Generating train split: 11514 examples [00:02, 4836.18 examples/s]
Generating validation split: 2033 examples [00:00, 2761.67 examples/s]
Generating test split: 2974 examples [00:00, 3496.53 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 22023.00 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 23787.78 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 22103.76 examples/s]
validation: 657 test: 914 utterances , unknown intents: 2060 utterances over 44 intents , far: 1509 utterances over 32 intents , near: 551 utterances ove

### Or from Python, to keep the results objects

In [ ]:
from experiments.ood_benchmark import run_known_benchmark

results, per_intent = run_known_benchmark(student_key="student1", keep=0.95)
per_intent.to_csv("outputs/results/ood_per_intent.csv")
results

### OOD benchmark against a V2 version of incremental trained model

In [3]:
from experiments.ood_benchmark import run_absorption_benchmark
import json

res = run_absorption_benchmark(
    checkpoint="outputs/checkpoints/student_ettin_68m_distill_v2",
    threshold=8.8752,
    new_intents=["car_window_control", "car_steering_wheel_heater_control"],
)
json.dump(res, open("outputs/results/absorption_v2_run3.json", "w"), indent=2)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

The repository for qanastek/MASSIVE contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/qanastek/MASSIVE.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


...

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/2974 [00:00<?, ? examples/s]

W1003 10:19:54.186000 2221 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode



student_ettin_68m_distill_v2: 17 intents, threshold 8.8752
caught far:  77.5% of 1509 utterances
caught near: 47.5% of 551 utterances

far-OOD utterances that passed: 339
  assigned to the 2 invented intents: 1.5% (they are 11.8% of the label space)

predicted intent of the far-OOD utterances that passed
                   utterances     share  invented
news_query                143  0.421829     False
play_music                 84  0.247788     False
datetime_query             26  0.076696     False
lists_createoradd          17  0.050147     False
weather_query              15  0.044248     False
audio_volume_mute          10  0.029499     False
transport_query            10  0.029499     False
alarm_set                   8  0.023599     False
play_radio                  6  0.017699     False
audio_volume_down           5  0.014749     False


In [4]:
from datasets import load_from_disk

for run in ["e2e_e2e_run1_20261002-132608", "e2e_e2e_run3_20261002-140003"]:
    ds = load_from_disk(f"outputs/results/{run}/dataset_v1")["train_set"]
    synth = ds.filter(lambda ex: str(ex["id"]).startswith("synth_"))
    print(f"\n=== {run}: {len(synth)} generated utterances for car_window_control")
    for u in synth["utt"][:40]:
        print(" ", u)

Filter:   0%|          | 0/3715 [00:00<?, ? examples/s]


=== e2e_e2e_run1_20261002-132608: 93 generated utterances for car_window_control
  open the window on the back driver side
  vent the car, open the windows slightly
  close the window on the right rear door
  raise the front passenger window
  close every window in the vehicle
  shut the driver side window now
  roll all the windows up now
  put the window down a few inches
  roll down every window
  windows down, it's nice outside
  car window down
  windows down on both sides
  drop all the windows down for a second
  open the right rear car window
  open the rear windshield window
  window up
  open the windows thirty percent
  open all the car windows for a minute
  put up all the windows before we park
  bring the passenger window down a little more
  windows up, we're going through the car wash
  lower the window on the driver's side please
  open the car window all the way
  lower the two front windows
  could you let down the back windows
  crack the driver window for some air

Filter:   0%|          | 0/3715 [00:00<?, ? examples/s]


=== e2e_e2e_run3_20261002-140003: 93 generated utterances for car_window_control
  shut my window please
  shut all four windows please
  roll down the window on the driver side
  bring the rear right window up
  please close the window on the passenger side
  close the rear left and right windows
  open the windows so the car can cool off
  open the rear driver side window
  raise every window in the car
  shut all windows and lock them
  drop all the windows for a second
  roll the passenger window down for the dog
  raise the passenger window it's windy
  open the windows halfway
  lower the rear passenger window please
  put up the back windows
  open the rear windows just a little
  close the sunroof and the windows
  bring the passenger window down
  crack the driver window for some air
  put the front passenger window up
  lower the front windows
  shut the windows before it rains
  lower the window so i can reach the ticket machine
  shut the passenger side windows
  put all w

In [6]:
from datasets import load_from_disk
ds = load_from_disk("outputs/results/e2e_e2e_run3_20261002-140003/dataset_v2")["train_set"]
synth = ds.filter(lambda ex: str(ex["id"]).startswith("synth_"))
hits = [u for u in synth["utt"] if "wheel" in u.lower()]
print(len(hits), "of", len(synth), "contain 'wheel'")
print(hits[:50])

95 of 188 contain 'wheel'
['my fingers are numb warm the steering wheel', 'crank up the heated wheel', 'steering wheel warmer medium please', 'heated wheel off', 'wheel heat level four', 'steering wheel heat setting three', 'can you warm the grip of the steering wheel', 'set wheel warmer to the lowest setting', 'warm the steering wheel until i say stop', 'turn the steering wheel heater on in the other car', 'full heat on the steering wheel', 'shut off the heated wheel', "steering wheel heat on please it's snowing", 'enable the wheel heater', 'the wheel heater is burning my hands lower it', 'keep the wheel heater off today', 'hey turn on heat for the steering wheel', 'put the wheel heater on automatic', 'heat my steering wheel for the commute', 'increase the heat on the steering wheel', 'please warm my steering wheel', 'steering wheel heat on auto', 'always turn on steering wheel heat when i start the car', 'set steering wheel warmer to minimum', 'could you make the steering wheel toast